# Detección y localización de grietas con aprendizaje débilmente supervisado

**Idea:** entrenamos un clasificador con etiquetas *de imagen completa* (grieta / sin grieta) y obtenemos la
**ubicación** de la grieta con **Grad-CAM**, sin haber usado nunca máscaras ni cajas durante el entrenamiento.
Eso es lo que hace al método *débilmente supervisado*.

**Cambios respecto a la versión anterior**

| # | Problema anterior | Corrección |
|---|---|---|
| 1 | Grad-CAM calculado sobre `modelo_base` (sin cabeza de clasificación) | Grad-CAM calculado sobre el logit de la cabeza `clasificador` |
| 2 | Faltaban `import numpy / cv2 / matplotlib` | Todos los imports en una sola celda |
| 3 | Clases desbalanceadas (15 % grietas) y solo se medía accuracy | Pesos por muestra + métricas AUC, precisión, recall, F1, matriz de confusión |
| 4 | Split aleatorio (recortes de la misma foto en train y val → fuga de datos) | Split **por foto de origen** en train / val / test |
| 5 | Base congelada, modelo subajustado | Fase 1 (cabeza) + Fase 2 (fine-tuning de los últimos bloques) |
| 6 | Mapa de calor de 8×8 (muy grueso para grietas finas) | Grad-CAM en 3 resoluciones (8×8, 16×16, 32×32) + fusión + binarización + cajas |
| 7 | Preprocesamiento `/255` | `preprocess_input` de MobileNetV2 (rango [-1, 1]) |
| 8 | Colores del overlay mezclaban BGR y RGB | Conversión correcta |
| 9 | Solo funcionaba con recortes de 256×256 | Localización en **imágenes grandes** con ventana deslizante |

> Ejecuta las celdas en orden. Usa un entorno con **GPU** (Entorno de ejecución → Cambiar tipo de entorno → GPU).


---

## Cambios de la **v3** (a partir de los resultados ejecutados de la v2)

La v2 dio AUC 0.929, precisión 0.890, **recall 0.630**, F1 0.738 en test. Se vio además que las cajas aparecían
en imágenes sin grieta y que el mapa de 32×32 era puro ruido de textura. Qué se cambió:

| # | Lo que se vio en la v2 | Cambio en la v3 |
|---|---|---|
| 1 | Recall 0.63: se pierde el 37 % de las grietas (el umbral 0.844 salió de maximizar F1) | El umbral se elige en validación con un **criterio configurable** (`CRITERIO_UMBRAL`: F1, F2 o recall objetivo) y se muestra una **tabla comparativa de los tres sobre test** |
| 2 | Solo se reportaba un subconjunto de métricas | Tabla completa: accuracy, accuracy balanceada, precisión, recall, especificidad, F1, F2, MCC, AUC-ROC, PR-AUC y TP/FP/FN/TN. Se exporta a CSV para comparar con otros modelos |
| 3 | Cajas dibujadas en imágenes **sin** grieta (el Grad-CAM se normaliza por su máximo, así que siempre resalta algo) | La máscara y las cajas **solo se generan si P(grieta) ≥ umbral** |
| 4 | El Grad-CAM de 32×32 es ruido de textura y contaminaba la fusión | La fusión usa solo 8×8 y 16×16; el de 32×32 queda como referencia visual |
| 5 | Máscaras en manchas y cajas fragmentadas sobre grietas finas | Refinamiento con un **mapa de crestas** (black-hat: estructuras oscuras y delgadas), filtrado por componentes conexas y cajas que unen fragmentos cercanos. Se apaga con `USAR_CRESTAS = False` |
| 6 | Predicciones sensibles a la orientación | **TTA** con giros (clasificador y Grad-CAM). Se apaga con `USAR_TTA = False` |
| 7 | El AUC de validación seguía subiendo en la época 10 (el entrenamiento paró por límite de épocas) | `MODO = 'continuar'`: **Fase 3** de fine-tuning con más bloques descongelados (desde `block_10`), monitoreando PR-AUC. `MODO = 'cargar'` evita reentrenar |
| 8 | Faltaba probar con imágenes de otra fuente | Sección 8b: `evaluar_externas()` sobre una carpeta con imágenes (por ejemplo de Google), con métricas si están etiquetadas |
| 9 | La localización no se medía | `evaluar_localizacion()` añade *pointing game* y permite comparar con/sin crestas |

> **Para no reentrenar** (lo más rápido): deja `MODO = 'cargar'`, ejecuta la celda de Drive, la de configuración y las
> secciones 1 a 3 (datos), y sigue desde la carga del modelo. El notebook ya trae el modelo guardado de la v2.

In [ ]:
# [SOLO_COLAB]
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, re, shutil, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.model_selection import GroupShuffleSplit
from IPython.display import display
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score, precision_score, recall_score,
                             accuracy_score, balanced_accuracy_score, f1_score, fbeta_score,
                             matthews_corrcoef)

# ---------------- Configuración ----------------
RUTA_DRIVE   = Path('/content/drive/MyDrive/IA 2/Dataset_IA2')   # dataset original
RUTA_LOCAL   = Path('/content/dataset')                          # copia local (más rápida que Drive)
RUTA_MODELOS = Path('/content/drive/MyDrive/IA 2/modelos')       # aquí se guardan los modelos
RUTA_MODELOS.mkdir(parents=True, exist_ok=True)

TAM_IMG  = (256, 256)
TAM_LOTE = 32
SEMILLA  = 123
EPOCAS_FASE1 = 5     # solo cabeza de clasificación
EPOCAS_FASE2 = 10    # fine-tuning
LR_FASE1 = 1e-3
LR_FASE2 = 1e-5

# ---------------- Modo de ejecución (v3) ----------------
#   'entrenar'  : Fase 1 + Fase 2 desde cero (~2 h en la GPU de Colab)
#   'cargar'    : carga el modelo ya entrenado y SOLO re-evalúa y localiza (minutos)
#   'continuar' : carga el modelo y hace una Fase 3 de fine-tuning (más bloques) -> modelo_grietas_v3.keras
MODO           = 'cargar'
ARCHIVO_MODELO = 'modelo_grietas_final.keras'   # modelo que se carga en 'cargar' / 'continuar' (luego puedes poner 'modelo_grietas_v3.keras')
EPOCAS_FASE3   = 12
LR_FASE3       = 1e-5
BLOQUE_FASE3   = 'block_10_expand'               # desde qué bloque se descongela en la Fase 3 (la Fase 2 usó block_13)

# ---------------- Decisión y localización (v3) ----------------
CRITERIO_UMBRAL = 'f2'      # cómo se elige el umbral en validación: 'f1' | 'f2' (prioriza recall) | 'recall' (recall objetivo)
RECALL_OBJETIVO = 0.80      # solo se usa si CRITERIO_UMBRAL = 'recall'
USAR_TTA        = True      # promedia predicciones y Grad-CAM de la imagen original y sus giros
USAR_CRESTAS    = True      # refina la máscara con el mapa de crestas (estructuras oscuras y delgadas)

random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    print('⚠️  No hay GPU: el entrenamiento será MUY lento. Activa una GPU en el entorno de ejecución.')

## 1. Organizar el dataset y registrar la foto de origen

Copiamos las imágenes a disco local (si no están ya) y construimos una tabla con la ruta, la etiqueta, la superficie
y un **identificador de grupo** (la foto de la que viene cada recorte). El dataset parece ser SDNET2018, donde cada
imagen de 256×256 es un recorte de una foto más grande y los nombres tienen la forma `7001-115.jpg`
(foto `7001`, recorte `115`). Si tus nombres son distintos, revisa la salida de diagnóstico de la celda.

In [ ]:
# [SOLO_COLAB]
from tqdm.auto import tqdm

SUPERFICIES = {'decks', 'pavements', 'walls'}
EXTENSIONES = {'.jpg', '.jpeg', '.png'}

def id_foto(nombre_archivo):
    """Extrae el id de la foto de origen: '7001-115.jpg' -> '7001'. Si no hay patrón, usa el nombre completo."""
    m = re.search(r'(\d+)[-_]\d+', Path(nombre_archivo).stem)
    return m.group(1) if m else Path(nombre_archivo).stem

registros = []
for carpeta_sup in sorted(RUTA_DRIVE.iterdir()):
    if not (carpeta_sup.is_dir() and carpeta_sup.name.lower() in SUPERFICIES):
        continue
    superficie = carpeta_sup.name
    for carpeta_cls in sorted(carpeta_sup.iterdir()):
        if not carpeta_cls.is_dir():
            continue
        nombre = carpeta_cls.name.lower().replace('-', '_')
        if nombre == 'cracked':
            etiqueta, destino = 1, 'grieta'
        elif nombre == 'non_cracked':
            etiqueta, destino = 0, 'sin_grieta'
        else:
            continue
        carpeta_dest = RUTA_LOCAL / destino
        carpeta_dest.mkdir(parents=True, exist_ok=True)
        archivos = [f for f in carpeta_cls.iterdir() if f.is_file() and f.suffix.lower() in EXTENSIONES]
        for f in tqdm(archivos, desc=f'{superficie}/{carpeta_cls.name}'):
            dest = carpeta_dest / f'{superficie}_{f.name}'
            if not dest.exists():
                shutil.copy(f, dest)
            registros.append({'ruta': str(dest), 'etiqueta': etiqueta, 'superficie': superficie,
                              'grupo': f'{superficie}_{id_foto(f.name)}'})

df = pd.DataFrame(registros)
df.to_csv(RUTA_LOCAL / 'indice.csv', index=False)

In [ ]:
print(f'Total de imágenes: {len(df)}')
print(df.groupby(['superficie', 'etiqueta']).size().unstack(fill_value=0).rename(columns={0: 'sin_grieta', 1: 'grieta'}))
print(f"\n% de imágenes con grieta: {100 * df.etiqueta.mean():.1f} %  "
      f"(un modelo que siempre diga 'sin grieta' ya tendría {100 * (1 - df.etiqueta.mean()):.1f} % de accuracy)")

# Diagnóstico del agrupamiento por foto
print(f"\nGrupos (fotos de origen) detectados: {df.grupo.nunique()}  ->  ~{len(df) / df.grupo.nunique():.1f} recortes por foto")
print('Ejemplos de nombres:', [Path(r).name for r in df.ruta.sample(5, random_state=SEMILLA)])
if df.grupo.nunique() > 0.8 * len(df):
    print('⚠️  Casi cada imagen es su propio grupo: el patrón de nombre no se detectó y el split NO evitará la fuga de datos.')

## 2. División train / validación / test **por foto de origen**

Si recortes de la misma foto caen en train y en validación, el modelo "reconoce" la foto y la métrica sale inflada.
Con `GroupShuffleSplit` todos los recortes de una misma foto quedan en un único conjunto.
El **test** no se toca hasta el final.

In [ ]:
def dividir_por_grupo(datos, fraccion, semilla):
    gss = GroupShuffleSplit(n_splits=1, test_size=fraccion, random_state=semilla)
    idx_a, idx_b = next(gss.split(datos, groups=datos['grupo']))
    return datos.iloc[idx_a].reset_index(drop=True), datos.iloc[idx_b].reset_index(drop=True)

df_trainval, df_test = dividir_por_grupo(df, 0.15, SEMILLA)
df_train, df_val     = dividir_por_grupo(df_trainval, 0.15 / 0.85, SEMILLA)

# Verificación: ningún grupo en dos conjuntos
assert not (set(df_train.grupo) & set(df_val.grupo)), 'Fuga train/val'
assert not (set(df_train.grupo) & set(df_test.grupo)), 'Fuga train/test'
assert not (set(df_val.grupo) & set(df_test.grupo)), 'Fuga val/test'

resumen = pd.DataFrame({
    nombre: {'imágenes': len(d), 'con grieta': int(d.etiqueta.sum()), '% grieta': round(100 * d.etiqueta.mean(), 1),
             'fotos (grupos)': d.grupo.nunique()}
    for nombre, d in [('train', df_train), ('val', df_val), ('test', df_test)]})
resumen

## 3. Pipeline de datos

- Aumento de datos **solo en entrenamiento** (giros, rotación, brillo, contraste). Una grieta sigue siendo grieta al girarla.
- `preprocess_input` de MobileNetV2 (rango [-1, 1]), que es lo que esperan los pesos de ImageNet.
- **Pesos por muestra** para compensar el desbalance: cada grieta pesa más que una imagen sin grieta.
- La mezcla (`shuffle`) se hace sobre las rutas, antes de cargar las imágenes, para no gastar memoria.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1, fill_mode='reflect'),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
], name='aumento_datos')

# Pesos para compensar el desbalance (n_total / (2 * n_clase))
n_total = len(df_train)
n1 = int(df_train.etiqueta.sum()); n0 = n_total - n1
PESO_CLASE = {0: n_total / (2 * n0), 1: n_total / (2 * n1)}
print('Pesos por clase:', {k: round(v, 2) for k, v in PESO_CLASE.items()})

def cargar_imagen(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, TAM_IMG)
    img.set_shape((*TAM_IMG, 3))
    return img, tf.cast(etiqueta, tf.float32)

def crear_dataset(datos, entrenamiento=False):
    rutas = datos['ruta'].astype(str).values
    etiquetas = datos['etiqueta'].values.astype('int32')
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if entrenamiento:
        ds = ds.shuffle(len(datos), seed=SEMILLA, reshuffle_each_iteration=True)
    ds = ds.map(cargar_imagen, num_parallel_calls=AUTOTUNE)
    if entrenamiento:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    if entrenamiento:
        w0, w1 = PESO_CLASE[0], PESO_CLASE[1]
        ds = ds.map(lambda x, y: (x, y, tf.where(y > 0.5, w1, w0)), num_parallel_calls=AUTOTUNE)
    return ds.batch(TAM_LOTE).prefetch(AUTOTUNE)

ds_train = crear_dataset(df_train, entrenamiento=True)
ds_val   = crear_dataset(df_val)
ds_test  = crear_dataset(df_test)

# Vista rápida de un lote de entrenamiento (con aumento de datos)
x, y, w = next(iter(ds_train))
fig, ejes = plt.subplots(2, 4, figsize=(14, 7))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow((x[i].numpy() + 1) / 2)          # de [-1,1] a [0,1] solo para mostrar
    eje.set_title(f"{'GRIETA' if y[i] > 0.5 else 'sin grieta'}  (peso {w[i]:.2f})")
    eje.axis('off')
plt.tight_layout(); plt.show()

## 4. Modelo: MobileNetV2 + `GlobalAveragePooling2D` + capa densa

`GlobalAveragePooling2D` conserva la relación entre cada canal del mapa de características y la decisión final,
que es lo que permite calcular el mapa de activación de clase. Las capas de la cabeza tienen nombre fijo
(`gap`, `dropout`, `clasificador`) para poder usarlas luego en Grad-CAM.

> La advertencia sobre `input_shape` y los pesos de 224×224 es normal: la red es totalmente convolucional y funciona con 256×256.

In [ ]:
def construir_modelo():
    base = keras.applications.MobileNetV2(input_shape=(*TAM_IMG, 3), include_top=False, weights='imagenet')
    base.trainable = False

    entradas = keras.Input(shape=(*TAM_IMG, 3), name='imagen')
    x = base(entradas, training=False)               # BatchNorm siempre en modo inferencia
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3, name='dropout')(x)
    salidas = layers.Dense(1, activation='sigmoid', name='clasificador')(x)
    return keras.Model(entradas, salidas, name='detector_grietas'), base

def compilar(modelo, lr):
    modelo.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=[keras.metrics.BinaryAccuracy(name='accuracy'),
                 keras.metrics.AUC(name='auc'),
                 keras.metrics.AUC(name='pr_auc', curve='PR'),      # más informativa que el AUC-ROC con clases desbalanceadas
                 keras.metrics.Precision(name='precision'),
                 keras.metrics.Recall(name='recall')])

def crear_callbacks(nombre, paciencia=3):
    return [
        keras.callbacks.EarlyStopping(monitor='val_pr_auc', mode='max', patience=paciencia, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_pr_auc', mode='max', factor=0.5, patience=1, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(str(RUTA_MODELOS / f'{nombre}.keras'), monitor='val_pr_auc',
                                        mode='max', save_best_only=True),
    ]

modelo, base = construir_modelo()
compilar(modelo, LR_FASE1)
modelo.summary()

In [ ]:
def obtener_base(modelo):
    """Devuelve el sub-modelo MobileNetV2 anidado dentro del modelo completo."""
    return next(l for l in modelo.layers if isinstance(l, keras.Model))

if MODO in ('cargar', 'continuar'):
    ruta_modelo = RUTA_MODELOS / ARCHIVO_MODELO
    if not ruta_modelo.exists():
        raise FileNotFoundError(f'No existe {ruta_modelo}. Ponle MODO = "entrenar" o corrige ARCHIVO_MODELO.')
    modelo = keras.models.load_model(ruta_modelo)
    base = obtener_base(modelo)
    print(f'Modelo cargado desde {ruta_modelo}  (MODO = {MODO!r})')
else:
    print("MODO = 'entrenar': se usa el modelo recién construido arriba.")

## 5. Entrenamiento

**Fase 1:** solo se entrena la cabeza (la base está congelada).
**Fase 2 (fine-tuning):** se descongelan los últimos bloques de MobileNetV2 (desde `block_13`) con una tasa de
aprendizaje muy baja. Las capas `BatchNormalization` se mantienen congeladas.
**Fase 3 (opcional, `MODO = 'continuar'`):** continúa desde el modelo guardado y descongela desde `block_10`.
En la v2 el AUC de validación todavía subía en la última época, así que había margen.
El mejor modelo se elige por **PR-AUC de validación** (más informativa que accuracy con clases desbalanceadas).

> Con `MODO = 'cargar'` las celdas de entrenamiento no hacen nada.

In [ ]:
if MODO == 'entrenar':
    print('=== FASE 1: entrenar la cabeza ===')
    hist1 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_FASE1,
                       callbacks=crear_callbacks('fase1_cabeza'))
else:
    print(f'MODO = {MODO!r}: se omite la Fase 1.')

In [ ]:
if MODO == 'entrenar':
    print('=== FASE 2: fine-tuning de los últimos bloques ===')
    base.trainable = True
    nombres_capas = [c.name for c in base.layers]
    inicio = nombres_capas.index('block_13_expand')      # primera capa del bloque 13
    for i, capa in enumerate(base.layers):
        capa.trainable = (i >= inicio) and not isinstance(capa, layers.BatchNormalization)
    print(f'Capas descongeladas: {sum(c.trainable for c in base.layers)} de {len(base.layers)}')

    compilar(modelo, LR_FASE2)   # hay que recompilar después de cambiar 'trainable'
    hist2 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_FASE2,
                       callbacks=crear_callbacks('fase2_finetuning'))

    modelo.save(RUTA_MODELOS / 'modelo_grietas_final.keras')
else:
    print(f'MODO = {MODO!r}: se omite la Fase 2.')

In [ ]:
if MODO == 'continuar':
    print(f'=== FASE 3: seguir el fine-tuning desde {BLOQUE_FASE3} ===')
    base = obtener_base(modelo)
    base.trainable = True
    nombres_capas = [c.name for c in base.layers]
    inicio = nombres_capas.index(BLOQUE_FASE3)
    for i, capa in enumerate(base.layers):
        capa.trainable = (i >= inicio) and not isinstance(capa, layers.BatchNormalization)
    print(f'Capas descongeladas: {sum(c.trainable for c in base.layers)} de {len(base.layers)}')

    compilar(modelo, LR_FASE3)
    hist3 = modelo.fit(ds_train, validation_data=ds_val, epochs=EPOCAS_FASE3,
                       callbacks=crear_callbacks('fase3_finetuning', paciencia=4))
    modelo.save(RUTA_MODELOS / 'modelo_grietas_v3.keras')      # no pisa el modelo de la v2
    print('Guardado: modelo_grietas_v3.keras  (para usarlo después: ARCHIVO_MODELO = "modelo_grietas_v3.keras")')
else:
    print(f'MODO = {MODO!r}: se omite la Fase 3.')

In [ ]:
# Curvas de entrenamiento (todas las fases que se ejecutaron en esta sesión)
hs = [globals()[n].history for n in ('hist1', 'hist2', 'hist3') if n in globals()]
if not hs:
    print(f'MODO = {MODO!r}: no hay historial de entrenamiento en esta sesión.')
else:
    h = {k: sum((x[k] for x in hs), []) for k in hs[0] if all(k in x for x in hs)}
    cortes, acumulado = [], 0
    for x in hs[:-1]:
        acumulado += len(x['loss']); cortes.append(acumulado - 0.5)
    fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
    for eje, (clave, titulo) in zip(ejes, [('loss', 'Pérdida'), ('auc', 'AUC'), ('recall', 'Recall (grietas encontradas)')]):
        eje.plot(h[clave], label='train'); eje.plot(h['val_' + clave], label='validación')
        for c in cortes:
            eje.axvline(c, color='gray', ls='--')
        eje.set_title(titulo); eje.set_xlabel('época'); eje.legend()
    plt.tight_layout(); plt.show()

## 6. Evaluación del clasificador

Con clases desbalanceadas la accuracy engaña (siempre decir "sin grieta" da ~84 %), así que se reportan **recall, precisión, F1 y MCC**.

El **umbral de decisión se elige con el conjunto de validación** y se aplica al **test**, que el modelo nunca vio.
En la v2 se usó el umbral que maximiza F1 (0.844) y salió recall = 0.63. Aquí se comparan tres criterios:

- **F1 óptimo:** equilibra precisión y recall (lo que hizo la v2).
- **F2 óptimo:** como F1 pero el recall pesa el doble que la precisión. Para inspección de estructuras, perder una grieta suele costar más que una falsa alarma.
- **Recall objetivo:** el umbral más alto que alcanza un recall dado en validación.

`CRITERIO_UMBRAL` (celda de configuración) decide cuál se usa en el resto del notebook; los otros quedan en la tabla para compararlos.

*Referencia v2 (test, umbral 0.844): accuracy 0.927 · precisión 0.890 · recall 0.630 · F1 0.738 · AUC-ROC 0.929 · PR-AUC 0.839.*

In [ ]:
def _giro_tf(ds, g):
    return ds.map(lambda x, y: (g(x), y))

def predecir(modelo, ds, tta=None):
    """Probabilidad de grieta por imagen. Con TTA promedia la imagen original y sus giros (H, V, H+V)."""
    tta = USAR_TTA if tta is None else tta
    giros = [lambda x: x]
    if tta:
        giros += [tf.image.flip_left_right, tf.image.flip_up_down,
                  lambda x: tf.image.flip_up_down(tf.image.flip_left_right(x))]
    return np.mean([modelo.predict(_giro_tf(ds, g), verbose=0).ravel() for g in giros], axis=0)

p_val  = predecir(modelo, ds_val)
p_test = predecir(modelo, ds_test)
y_val, y_test = df_val.etiqueta.values, df_test.etiqueta.values

def umbral_por_criterio(y, p, criterio):
    prec, rec, umbrales = precision_recall_curve(y, p)
    prec, rec = prec[:-1], rec[:-1]                         # alinear con 'umbrales'
    if criterio in ('f1', 'f2'):
        b2 = 1.0 if criterio == 'f1' else 4.0               # beta^2
        f = (1 + b2) * prec * rec / (b2 * prec + rec + 1e-8)
        return float(umbrales[np.argmax(f)])
    if criterio == 'recall':
        ok = np.where(rec >= RECALL_OBJETIVO)[0]            # el recall baja al subir el umbral
        return float(umbrales[ok.max()]) if len(ok) else float(umbrales[0])
    raise ValueError("CRITERIO_UMBRAL debe ser 'f1', 'f2' o 'recall'")

def metricas_completas(y, pred, score=None):
    """pred: 0/1 por imagen. score (opcional): probabilidad, para AUC-ROC y PR-AUC."""
    y = np.asarray(y).astype(int); pred = np.asarray(pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    m = {'accuracy': accuracy_score(y, pred),
         'accuracy balanceada': balanced_accuracy_score(y, pred),
         'precisión': precision_score(y, pred, zero_division=0),
         'recall': recall_score(y, pred, zero_division=0),
         'especificidad': tn / (tn + fp) if (tn + fp) else np.nan,
         'F1': f1_score(y, pred, zero_division=0),
         'F2': fbeta_score(y, pred, beta=2, zero_division=0),
         'MCC': matthews_corrcoef(y, pred)}
    if score is not None and len(np.unique(y)) == 2:
        m['AUC-ROC'] = roc_auc_score(y, score); m['PR-AUC'] = average_precision_score(y, score)
    m.update({'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)})
    return {k: (round(float(v), 3) if isinstance(v, float) else v) for k, v in m.items()}

# --- Tres criterios de umbral, elegidos en VALIDACIÓN y evaluados en TEST ---
CRITERIOS = {'F1 óptimo': 'f1', 'F2 óptimo (prioriza recall)': 'f2', f'Recall ≥ {RECALL_OBJETIVO:.0%} en val': 'recall'}
umbrales = {nombre: umbral_por_criterio(y_val, p_val, c) for nombre, c in CRITERIOS.items()}
filas = {nombre: {'umbral': round(u, 3), **metricas_completas(y_test, p_test >= u, p_test)} for nombre, u in umbrales.items()}
filas['Umbral fijo 0.5 (referencia)'] = {'umbral': 0.5, **metricas_completas(y_test, p_test >= 0.5, p_test)}
tabla_metricas = pd.DataFrame(filas).T
tabla_metricas[['TP', 'FP', 'FN', 'TN']] = tabla_metricas[['TP', 'FP', 'FN', 'TN']].astype(int)
print('Métricas en TEST según el criterio de umbral (el umbral se eligió en validación):')
display(tabla_metricas)

nombre_elegido = next(k for k, v in CRITERIOS.items() if v == CRITERIO_UMBRAL)
UMBRAL_DECISION = umbrales[nombre_elegido]
pred_test = (p_test >= UMBRAL_DECISION).astype(int)
print(f'\n>>> Criterio en uso: {nombre_elegido}  ->  UMBRAL_DECISION = {UMBRAL_DECISION:.3f}\n')
print(classification_report(y_test, pred_test, target_names=['sin_grieta', 'grieta'], digits=3))
print(f'AUC-ROC test: {roc_auc_score(y_test, p_test):.3f}   |   Average Precision (PR-AUC): {average_precision_score(y_test, p_test):.3f}')
print(f'Línea base "siempre sin grieta": accuracy = {100 * (1 - y_test.mean()):.1f} %')

archivo_csv = RUTA_MODELOS / f'metricas_test_{Path(ARCHIVO_MODELO).stem}.csv'
tabla_metricas.to_csv(archivo_csv, encoding='utf-8-sig')
print(f'Tabla guardada en {archivo_csv}  (para compararla con las de los otros modelos)')

fig, ejes = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_test), display_labels=['sin_grieta', 'grieta']).plot(ax=ejes[0], colorbar=False)
ejes[0].set_title(f'Matriz de confusión (test) · umbral {UMBRAL_DECISION:.2f}')
fpr, tpr, _ = roc_curve(y_test, p_test)
ejes[1].plot(fpr, tpr); ejes[1].plot([0, 1], [0, 1], 'k--'); ejes[1].set_title('Curva ROC'); ejes[1].set_xlabel('FPR'); ejes[1].set_ylabel('TPR')
p_, r_, _ = precision_recall_curve(y_test, p_test)
ejes[2].plot(r_, p_, label='curva PR')
for nombre, fila in filas.items():
    ejes[2].scatter(fila['recall'], fila['precisión'], s=50, zorder=3, label=nombre)
ejes[2].set_title('Curva Precisión-Recall y puntos de operación'); ejes[2].set_xlabel('Recall'); ejes[2].set_ylabel('Precisión')
ejes[2].legend(fontsize=7, loc='lower left')
plt.tight_layout(); plt.show()

In [ ]:
# Rendimiento por tipo de superficie (test), con el umbral en uso
filas = []
for sup, g in df_test.assign(prob=p_test).groupby('superficie'):
    pred = (g.prob >= UMBRAL_DECISION).astype(int)
    filas.append({'superficie': sup, 'imágenes': len(g), '% grieta': round(100 * g.etiqueta.mean(), 1),
                  'recall': round(recall_score(g.etiqueta, pred, zero_division=0), 3),
                  'precisión': round(precision_score(g.etiqueta, pred, zero_division=0), 3),
                  'F1': round(f1_score(g.etiqueta, pred, zero_division=0), 3),
                  'AUC': round(roc_auc_score(g.etiqueta, g.prob), 3) if g.etiqueta.nunique() == 2 else None})
pd.DataFrame(filas)

## 7. Localización con Grad-CAM (v3)

**Base (igual que v2):** el gradiente se calcula respecto al **logit de la clase "grieta"** de la cabeza `clasificador`
(usar el logit evita gradientes nulos por saturación de la sigmoide), en capas de 8×8, 16×16 y 32×32.

**Qué cambió y por qué (a partir de lo que se vio en las imágenes de la v2):**

1. **Solo se dibuja máscara y cajas si P(grieta) ≥ umbral.** El Grad-CAM se normaliza por su propio máximo, así que en una
   imagen sin grieta siempre "resalta algo" y se dibujaban cajas sobre ruido. Ahora el mapa se muestra solo como diagnóstico.
2. **El mapa de 32×32 ya no entra en la fusión.** En las figuras era una nube de puntos de textura, también en imágenes
   con grieta clara. La fusión (media geométrica) usa 8×8 (dónde mirar) y 16×16 (la forma). El de 32×32 se sigue mostrando como referencia.
3. **Refinamiento con crestas** (`USAR_CRESTAS`): el mapa de calor se multiplica por una respuesta *black-hat*, que resalta estructuras
   oscuras y delgadas (lo que es una grieta). El mapa dice *dónde*, la respuesta de crestas dice *qué píxeles*. Así la máscara se
   acerca a la forma de la grieta en vez de ser una mancha. Limitación: asume grietas más oscuras que su entorno.
4. **TTA** (`USAR_TTA`): se promedian el mapa y la probabilidad de la imagen original y sus giros, lo que reduce el ruido.
5. **Cajas:** la máscara se limpia por componentes conexas (en vez de una apertura 5×5 que borraría grietas finas) y las cajas
   unen fragmentos cercanos, para tener una caja por grieta y no una por trozo.

In [ ]:
def obtener_base(modelo):
    """Devuelve el sub-modelo MobileNetV2 anidado dentro del modelo completo."""
    return next(l for l in modelo.layers if isinstance(l, keras.Model))

_extractores = {}
def _extractor(modelo, nombre_capa):
    clave = (id(modelo), nombre_capa)
    if clave not in _extractores:
        b = obtener_base(modelo)
        _extractores[clave] = keras.Model(b.input, [b.get_layer(nombre_capa).output, b.output])
    return _extractores[clave]

def grad_cam(img_prep, modelo, nombre_capa='out_relu'):
    """
    img_prep: array (1, 256, 256, 3) ya pasado por preprocess_input.
    Devuelve (mapa_calor en [0,1] con la resolución de la capa, probabilidad de grieta).
    """
    extractor = _extractor(modelo, nombre_capa)
    gap, clasif = modelo.get_layer('gap'), modelo.get_layer('clasificador')
    x = tf.convert_to_tensor(img_prep)
    with tf.GradientTape() as cinta:
        cinta.watch(x)                                   # asegura que se registre toda la cadena
        act_capa, mapas_finales = extractor(x, training=False)
        vector = gap(mapas_finales)
        logit = tf.matmul(vector, clasif.kernel) + clasif.bias   # logit de "grieta"
        objetivo = logit[:, 0]
    grads = cinta.gradient(objetivo, act_capa)
    pesos = tf.reduce_mean(grads, axis=(0, 1, 2))               # importancia de cada canal
    cam = tf.nn.relu(tf.reduce_sum(act_capa[0] * pesos, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return cam, float(tf.sigmoid(logit)[0, 0])

# Capas disponibles (resolución para entrada de 256×256) y cuáles entran en la fusión
CAPAS_CAM = {'8×8 (out_relu)': 'out_relu',
             '16×16 (block_13_expand_relu)': 'block_13_expand_relu',
             '32×32 (block_6_expand_relu)': 'block_6_expand_relu'}
CAPAS_FUSION = ['out_relu', 'block_13_expand_relu']      # el de 32×32 es ruido de textura: solo referencia visual

# Giros para TTA: ejes del lote (1, H, W, 3) que se invierten. Cada giro es su propia inversa.
GIROS_TTA = [(), (2,), (1,), (1, 2)]

def cam_tta(img_prep, modelo, nombre_capa, tta=True):
    """Grad-CAM promediado sobre la imagen y sus giros (cada mapa se des-gira antes de promediar)."""
    cams, probs = [], []
    for ejes in (GIROS_TTA if tta else GIROS_TTA[:1]):
        x = np.ascontiguousarray(np.flip(img_prep, axis=ejes)) if ejes else img_prep
        cam, p = grad_cam(x, modelo, nombre_capa)
        ejes_cam = tuple(a - 1 for a in ejes)                    # (1, H, W, 3) -> (H, W)
        cams.append(np.flip(cam, axis=ejes_cam) if ejes_cam else cam)
        probs.append(p)
    cam = np.mean(cams, axis=0)
    return cam / (cam.max() + 1e-8), float(np.mean(probs))

def redimensionar_cam(cam, tam=TAM_IMG):
    return np.clip(cv2.resize(cam, (tam[1], tam[0]), interpolation=cv2.INTER_CUBIC), 0, 1)

def fusionar_cams(lista_cams):
    """Media geométrica de mapas ya redimensionados a 256×256."""
    f = np.exp(np.mean(np.log(np.stack(lista_cams) + 1e-3), axis=0))
    return (f - f.min()) / (f.max() - f.min() + 1e-8)

def mapa_crestas(img, escala_gris=50.0, ancho_max=15):
    """Respuesta a estructuras oscuras y delgadas (black-hat). img: RGB uint8. Devuelve valores en [0,1].
    La escala es ABSOLUTA (no por imagen): la textura del concreto da valores bajos y una grieta oscura, altos."""
    gris = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), (3, 3), 0)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ancho_max, ancho_max))
    bh = cv2.morphologyEx(gris, cv2.MORPH_BLACKHAT, k).astype(np.float32)
    return np.clip(bh / escala_gris, 0, 1)

def refinar_con_crestas(cam, img, peso_cam=0.25):
    """El mapa de calor dice DÓNDE mirar; las crestas dicen QUÉ píxeles son una línea oscura."""
    crestas = cv2.dilate(mapa_crestas(img), np.ones((3, 3), np.uint8))     # tolera pequeños desalineamientos
    ref = cam * (peso_cam + (1 - peso_cam) * crestas)
    return ref / (ref.max() + 1e-8)

def cam_fusionada(img, modelo, tta=None, crestas=None, capas_extra=False):
    """
    img: imagen RGB uint8 de 256×256.
    Devuelve (mapa fusionado en [0,1], {capa: mapa 256×256}, probabilidad de grieta).
    capas_extra=True calcula también el mapa de 32×32 (solo para mostrarlo).
    """
    tta = USAR_TTA if tta is None else tta
    crestas = USAR_CRESTAS if crestas is None else crestas
    prep = preprocess_input(img[None].astype('float32'))
    capas = list(CAPAS_CAM.values()) if capas_extra else CAPAS_FUSION
    cams, prob = {}, None
    for capa in capas:
        cam, p = cam_tta(prep, modelo, capa, tta)
        cams[capa] = redimensionar_cam(cam)
        prob = p if prob is None else prob
    fusion = fusionar_cams([cams[c] for c in CAPAS_FUSION])
    if crestas:
        fusion = refinar_con_crestas(fusion, img)
    return fusion, cams, prob

def binarizar(cam, umbral_cam=0.5, area_min_px=40):
    """Umbral + cierre mínimo + descarte de componentes pequeñas (no se usa apertura: borraría grietas finas)."""
    mascara = (cam >= umbral_cam).astype(np.uint8)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(mascara, connectivity=8)
    limpia = np.zeros_like(mascara)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= area_min_px:
            limpia[etiquetas == i] = 1
    return limpia

def obtener_cajas(mascara, area_minima=80, unir=15):
    """Cajas (x, y, w, h). 'unir' es el tamaño del cierre que junta fragmentos cercanos en una sola caja."""
    unida = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (unir, unir)))
    contornos, _ = cv2.findContours(unida, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cajas = []
    for c in contornos:
        x, y, w, h = cv2.boundingRect(c)
        if mascara[y:y + h, x:x + w].sum() >= area_minima:        # píxeles de máscara reales dentro de la caja
            cajas.append((x, y, w, h))
    return cajas

def superponer(img, cam, alpha=0.4):
    color = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img, 1 - alpha, color, alpha, 0)

def pintar_resultado(img, mascara, cajas, grosor=2):
    """Máscara en rojo translúcido y cajas en verde."""
    out = img.copy()
    if mascara.any():
        m = mascara.astype(bool)
        rojo = np.array([255, 0, 0], dtype=np.float32)
        out[m] = (0.45 * out[m] + 0.55 * rojo).astype(np.uint8)
    for (x, y, w, h) in cajas:
        cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), grosor)
    return out

# Comprobación de resoluciones reales de cada capa
b = obtener_base(modelo)
for titulo, capa in CAPAS_CAM.items():
    print(f'{capa:25s} -> salida {b.get_layer(capa).output.shape[1:]}')

In [ ]:
def visualizar(ruta, modelo, umbral_cam=0.5, crestas=None, mostrar=True):
    img = np.array(keras.utils.load_img(ruta, target_size=TAM_IMG))            # RGB uint8
    fusion, cams, prob = cam_fusionada(img, modelo, crestas=crestas, capas_extra=True)
    es_grieta = prob >= UMBRAL_DECISION

    # La máscara y las cajas solo existen si el clasificador dice "grieta"
    mascara = binarizar(fusion, umbral_cam) if es_grieta else np.zeros(TAM_IMG, np.uint8)
    cajas = obtener_cajas(mascara) if es_grieta else []

    usa_crestas = USAR_CRESTAS if crestas is None else crestas
    paneles = [(img, 'Original')]
    for titulo, capa in CAPAS_CAM.items():
        extra = '  (solo referencia)' if capa not in CAPAS_FUSION else ''
        paneles.append((superponer(img, cams[capa]), f'Grad-CAM {titulo.split(" ")[0]}{extra}'))
    # el orden queda: original, 8×8, 16×16, 32×32, fusión, resultado
    paneles.append((superponer(img, fusion), 'Fusión 8×8 · 16×16' + (' + crestas' if usa_crestas else '')))
    paneles.append((pintar_resultado(img, mascara, cajas),
                    f'Máscara ≥ {umbral_cam} + {len(cajas)} caja(s)' if es_grieta else 'P < umbral: sin máscara ni cajas'))

    fig, ejes = plt.subplots(2, 3, figsize=(14, 9.5))
    for eje, (im, titulo) in zip(ejes.ravel(), paneles):
        eje.imshow(im); eje.set_title(titulo); eje.axis('off')
    veredicto = 'GRIETA' if es_grieta else 'sin grieta'
    fig.suptitle(f'{Path(ruta).name}   |   P(grieta) = {prob:.3f}  (umbral {UMBRAL_DECISION:.3f})  ->  {veredicto}', fontsize=13)
    plt.tight_layout()
    if mostrar:
        plt.show()
    else:
        plt.close(fig)
    return fusion, mascara, cajas, prob

# Ejemplos del conjunto de test: aciertos, falsos negativos y falsos positivos
df_ej = df_test.assign(prob=p_test)
grupos_ej = [
    ('ACIERTOS (grietas detectadas con más confianza)',
     df_ej[(df_ej.etiqueta == 1) & (df_ej.prob >= UMBRAL_DECISION)].sort_values('prob', ascending=False).head(3)),
    ('FALSOS NEGATIVOS (grieta que el modelo no vio)',
     df_ej[(df_ej.etiqueta == 1) & (df_ej.prob < UMBRAL_DECISION)].sort_values('prob', ascending=False).head(2)),
    ('FALSOS POSITIVOS (sin grieta, pero el modelo dijo grieta)',
     df_ej[(df_ej.etiqueta == 0) & (df_ej.prob >= UMBRAL_DECISION)].head(2)),
    ('VERDADEROS NEGATIVOS (ya no deben dibujar cajas)',
     df_ej[(df_ej.etiqueta == 0) & (df_ej.prob < UMBRAL_DECISION)].head(2)),
]
for titulo, g in grupos_ej:
    print('\n' + '=' * 90 + f'\n{titulo}\n' + '=' * 90)
    for ruta in g.ruta:
        visualizar(ruta, modelo)

## 8. Localizar grietas en **cualquier** imagen (ventana deslizante)

El modelo se entrenó con recortes de 256×256. Si le das una foto grande de una pared, reducirla a 256×256 haría
desaparecer las grietas finas. Por eso recorremos la foto con ventanas de 256×256 (con solapamiento), clasificamos
cada ventana y aplicamos Grad-CAM solo en las que el modelo marca como grieta. Luego se recompone el mapa completo.

Parámetros útiles:
- `escala`: factor para redimensionar la foto antes de recorrerla. Los recortes de entrenamiento muestran la grieta de cerca;
  en una foto de baja resolución donde la pared aparece completa, prueba `escala=2.0` para que la grieta tenga un grosor parecido.
- `min_ventanas`: cuántas ventanas deben superar el umbral para declarar grieta (con `paso=128` las ventanas se solapan al 50 %,
  y una grieta real suele activar varias; subirlo a 2 reduce falsas alarmas).

Uso: `resultado = localizar_en_imagen_grande('/content/drive/MyDrive/mi_foto.jpg', modelo)`

In [ ]:
def preparar_imagen_grande(ruta, escala=1.0):
    bgr = cv2.imread(str(ruta))
    if bgr is None:
        raise FileNotFoundError(f'No se pudo leer la imagen: {ruta}')
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    t = TAM_IMG[0]
    if escala != 1.0:
        img = cv2.resize(img, None, fx=escala, fy=escala,
                         interpolation=cv2.INTER_CUBIC if escala > 1 else cv2.INTER_AREA)
    H, W = img.shape[:2]
    if min(H, W) < t:                                            # imagen más pequeña que una ventana
        esc = t / min(H, W)
        img = cv2.resize(img, None, fx=esc, fy=esc, interpolation=cv2.INTER_CUBIC)
    return img

def coordenadas_ventanas(H, W, paso=128):
    t = TAM_IMG[0]
    ys = list(range(0, H - t + 1, paso)); xs = list(range(0, W - t + 1, paso))
    if ys[-1] != H - t: ys.append(H - t)
    if xs[-1] != W - t: xs.append(W - t)
    return [(y, x) for y in ys for x in xs]

def probabilidades_ventanas(img, coords, modelo, tta=None):
    """P(grieta) de cada ventana (por lotes para no agotar memoria). Con TTA promedia los giros."""
    tta = USAR_TTA if tta is None else tta
    t = TAM_IMG[0]
    probs = []
    for i in range(0, len(coords), 64):
        lote = np.stack([img[y:y + t, x:x + t] for y, x in coords[i:i + 64]]).astype('float32')
        lote = preprocess_input(lote)
        variantes = [lote] + ([lote[:, :, ::-1], lote[:, ::-1], lote[:, ::-1, ::-1]] if tta else [])
        probs.append(np.mean([modelo.predict(np.ascontiguousarray(v), verbose=0).ravel() for v in variantes], axis=0))
    return np.concatenate(probs)

def localizar_en_imagen_grande(ruta, modelo, paso=128, umbral_prob=None, umbral_cam=0.5, max_parches_cam=200,
                               min_ventanas=1, escala=1.0, tta_cam=False, mostrar=True):
    umbral_prob = UMBRAL_DECISION if umbral_prob is None else umbral_prob
    img = preparar_imagen_grande(ruta, escala)
    H, W = img.shape[:2]
    t = TAM_IMG[0]
    coords = coordenadas_ventanas(H, W, paso)
    if mostrar:
        print(f'Imagen {W}×{H}px  ->  {len(coords)} ventanas de {t}×{t}')

    # 1) Probabilidad de grieta por ventana
    probs = probabilidades_ventanas(img, coords, modelo)
    mapa_prob = np.zeros((H, W), np.float32); cont = np.zeros((H, W), np.float32)
    for (y, x), p in zip(coords, probs):
        mapa_prob[y:y + t, x:x + t] += p; cont[y:y + t, x:x + t] += 1
    mapa_prob /= np.maximum(cont, 1)

    # 2) Grad-CAM fino solo en las ventanas positivas (las más seguras primero)
    n_positivas = int((probs >= umbral_prob).sum())
    hay_grieta = n_positivas >= min_ventanas
    candidatas = [i for i in np.argsort(-probs) if probs[i] >= umbral_prob][:max_parches_cam] if hay_grieta else []
    acum = np.zeros((H, W), np.float32)
    for i in candidatas:
        y, x = coords[i]
        fusion, _, _ = cam_fusionada(img[y:y + t, x:x + t], modelo, tta=tta_cam)
        acum[y:y + t, x:x + t] += fusion * probs[i]
    mapa_cam = acum / np.maximum(cont, 1)
    if mapa_cam.max() > 0:
        mapa_cam /= mapa_cam.max()

    mascara = binarizar(mapa_cam, umbral_cam, area_min_px=150) if candidatas else np.zeros((H, W), np.uint8)
    cajas = obtener_cajas(mascara, area_minima=300, unir=31) if candidatas else []

    if mostrar:
        fig, ejes = plt.subplots(1, 3, figsize=(18, 6))
        ejes[0].imshow(img); ejes[0].set_title('Original')
        ejes[1].imshow(superponer(img, mapa_prob)); ejes[1].set_title(f'P(grieta) por ventana · {n_positivas}/{len(coords)} sobre el umbral')
        ejes[2].imshow(pintar_resultado(img, mascara, cajas, grosor=max(2, H // 300)))
        ejes[2].set_title(f'Localización: {len(cajas)} zona(s) con grieta' if cajas else 'No se detectaron grietas')
        for e in ejes: e.axis('off')
        plt.tight_layout(); plt.show()
    return {'mapa_prob': mapa_prob, 'mapa_cam': mapa_cam, 'mascara': mascara, 'cajas': cajas,
            'prob_max': float(probs.max()), 'n_positivas': n_positivas, 'n_ventanas': len(coords)}

# Ejemplo (cambia la ruta por una foto tuya):
# resultado = localizar_en_imagen_grande('/content/drive/MyDrive/IA 2/mi_foto_pared.jpg', modelo)

## 8b. Probar con imágenes de **otra fuente** (por ejemplo, de Google)

Sirve para ver si el modelo generaliza fuera de SDNET2018. Organiza una carpeta en Drive así:

```
imagenes_externas/
    grieta/       <- fotos que SÍ tienen grieta
    sin_grieta/   <- fotos de superficies sin grieta (¡importantes para medir falsas alarmas!)
```

Si pones las imágenes sueltas en la carpeta (sin subcarpetas) solo se muestran predicciones, sin métricas.

Cada foto se recorre con la ventana deslizante y se decide con dos reglas: grieta si **≥ 1** ventana supera el umbral, o si **≥ 2** lo hacen.
Con muy pocas imágenes las métricas son orientativas y **no son comparables con las del test**, porque cambian la cámara, la distancia y la resolución.
Si las fotos son pequeñas o la pared aparece completa, prueba `escala=2.0`.

In [ ]:
from tqdm.auto import tqdm

def evaluar_externas(carpeta, modelo, paso=128, escala=1.0, reglas=(1, 2), mostrar_n=0, guardar_csv=True):
    """
    carpeta/grieta/*.jpg y carpeta/sin_grieta/*.jpg -> predicciones + métricas.
    carpeta/*.jpg (sueltas)                         -> solo predicciones.
    mostrar_n: cuántas imágenes dibujar con localización (las de mayor probabilidad primero).
    """
    carpeta = Path(carpeta)
    exts = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    archivos = []
    for f in sorted(carpeta.rglob('*')):
        if f.is_file() and f.suffix.lower() in exts:
            padre = f.parent.name.lower().replace('-', '_')
            etiqueta = 1 if padre == 'grieta' else 0 if padre == 'sin_grieta' else None
            archivos.append((f, etiqueta))
    if not archivos:
        raise FileNotFoundError(f'No hay imágenes en {carpeta}')

    filas = []
    for f, etiqueta in tqdm(archivos, desc='imágenes externas'):
        img = preparar_imagen_grande(f, escala)
        coords = coordenadas_ventanas(*img.shape[:2], paso)
        probs = probabilidades_ventanas(img, coords, modelo)
        n_pos = int((probs >= UMBRAL_DECISION).sum())
        fila = {'imagen': f.name, 'etiqueta': etiqueta, 'ventanas': len(coords),
                'P máx': round(float(probs.max()), 3), 'ventanas positivas': n_pos}
        for k in reglas:
            fila[f'pred (≥{k} ventana)'] = int(n_pos >= k)
        filas.append(fila)
    res = pd.DataFrame(filas)
    display(res)

    etiquetadas = res.dropna(subset=['etiqueta'])
    if len(etiquetadas) == len(res) and res.etiqueta.nunique() == 2:
        y = res.etiqueta.astype(int).values
        tabla = {f'Regla: ≥{k} ventana(s)': metricas_completas(y, res[f'pred (≥{k} ventana)'].values, res['P máx'].values)
                 for k in reglas}
        print(f'\nMétricas en imágenes externas  (n = {len(res)}: {int(y.sum())} con grieta, {int((1 - y).sum())} sin grieta)')
        t = pd.DataFrame(tabla).T
        t[['TP', 'FP', 'FN', 'TN']] = t[['TP', 'FP', 'FN', 'TN']].astype(int)
        display(t)
    elif len(etiquetadas):
        print('\nHay imágenes etiquetadas y sin etiquetar mezcladas, o solo una clase: no se calculan métricas.')
    if guardar_csv:
        salida = RUTA_MODELOS / f'externas_{Path(ARCHIVO_MODELO).stem}.csv'
        res.to_csv(salida, index=False, encoding='utf-8-sig'); print(f'Resultados guardados en {salida}')

    for nombre in res.sort_values('P máx', ascending=False).imagen.head(mostrar_n):
        ruta = next(f for f, _ in archivos if f.name == nombre)
        print(f'\n--- {nombre} ---')
        localizar_en_imagen_grande(ruta, modelo, paso=paso, escala=escala)
    return res

# Ejemplo (ajusta la ruta):
# RUTA_EXTERNAS = Path('/content/drive/MyDrive/IA 2/imagenes_externas')
# res_ext = evaluar_externas(RUTA_EXTERNAS, modelo, mostrar_n=4)

## 9. ¿Qué tan buena es la localización?

Como el entrenamiento no usa máscaras, la calidad de la localización **no se puede medir con el dataset original**.
Para poder reportarla, anota a mano una muestra pequeña (30–50 imágenes con grieta del *test*) con una herramienta
como LabelMe, CVAT o Roboflow, exporta máscaras binarias en PNG (blanco = grieta) y usa la función de abajo.

Dos advertencias para interpretar los números:
- Una grieta es una línea de pocos píxeles y el mapa de calor es una mancha, así que el IoU tenderá a ser bajo
  incluso si la ubicación es correcta. Por eso la función dilata un poco la máscara real (`dilatar_gt`) y también
  reporta qué fracción de la grieta real queda cubierta (`cobertura`) y el *pointing game* (`punto_max_en_grieta`:
  si el punto más caliente del mapa cae sobre la grieta real; es la métrica estándar en localización débilmente supervisada).
- La localización se mide **sin** el filtro de P(grieta): así se evalúa el mapa por sí solo. La columna `detectada` indica si
  el clasificador la habría marcado como grieta, y se reporta también el promedio solo sobre las detectadas.
- Reporta también el caso "sin grieta" (falsos positivos), no solo los aciertos.

Para el informe, lo más convincente es **comparar con y sin crestas** (ver el ejemplo de uso)

In [ ]:
def calcular_metricas_localizacion(mascara_pred, mascara_gt):
    p, g = mascara_pred.astype(bool), mascara_gt.astype(bool)
    inter = np.logical_and(p, g).sum(); union = np.logical_or(p, g).sum()
    return {'IoU': inter / union if union else 1.0,
            'Dice': 2 * inter / (p.sum() + g.sum()) if (p.sum() + g.sum()) else 1.0,
            'cobertura': inter / g.sum() if g.sum() else np.nan}      # % de la grieta real cubierta

def evaluar_localizacion(pares, modelo, umbral_cam=0.5, dilatar_gt=9, crestas=None, verbose=True):
    """pares: lista de (ruta_imagen_256, ruta_mascara_png)."""
    filas = []
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilatar_gt, dilatar_gt))
    for ruta_img, ruta_mask in pares:
        img = np.array(keras.utils.load_img(ruta_img, target_size=TAM_IMG))
        fusion, _, prob = cam_fusionada(img, modelo, crestas=crestas)
        pred = binarizar(fusion, umbral_cam)
        gt = cv2.imread(str(ruta_mask), cv2.IMREAD_GRAYSCALE)
        gt = (cv2.resize(gt, (TAM_IMG[1], TAM_IMG[0]), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)
        gt = cv2.dilate(gt, k)
        pico = np.unravel_index(fusion.argmax(), fusion.shape)
        filas.append({'imagen': Path(ruta_img).name, 'prob': round(prob, 3), 'detectada': prob >= UMBRAL_DECISION,
                      'punto_max_en_grieta': float(gt[pico] > 0), **calcular_metricas_localizacion(pred, gt)})
    res = pd.DataFrame(filas)
    cols = ['IoU', 'Dice', 'cobertura', 'punto_max_en_grieta']
    if verbose:
        print(f'Todas las imágenes (n = {len(res)}):'); print(res[cols].mean().round(3).to_string())
        det = res[res.detectada]
        print(f'\nSolo las detectadas por el clasificador (n = {len(det)}):'); print(det[cols].mean().round(3).to_string())
    return res

# Ejemplo de uso cuando tengas tus máscaras:
# pares = [('/content/dataset/grieta/Walls_7001-115.jpg', '/content/drive/MyDrive/IA 2/mascaras/Walls_7001-115.png'), ...]
# sin_c = evaluar_localizacion(pares, modelo, crestas=False, verbose=False)
# con_c = evaluar_localizacion(pares, modelo, crestas=True,  verbose=False)
# print(pd.DataFrame({'sin crestas': sin_c[['IoU','Dice','cobertura','punto_max_en_grieta']].mean(),
#                     'con crestas': con_c[['IoU','Dice','cobertura','punto_max_en_grieta']].mean()}).round(3))

## 10. Cargar el modelo guardado más adelante

```python
modelo = keras.models.load_model('/content/drive/MyDrive/IA 2/modelos/modelo_grietas_final.keras')   # o modelo_grietas_v3.keras
UMBRAL_DECISION = 0.5   # usa el valor que salió en la sección 6 (el del criterio en uso)
```

## Resumen para la defensa

- **Supervisión débil:** solo se usan etiquetas por imagen (grieta / sin grieta); nunca se entrena con coordenadas ni máscaras.
- **Validación honesta:** split por foto de origen, umbral elegido en validación, test intacto hasta el final.
- **Métricas para clases desbalanceadas:** recall, precisión, F1/F2, MCC y accuracy balanceada, no solo accuracy. El umbral es una decisión
  de diseño: priorizar recall (F2) reduce las grietas perdidas a cambio de más falsas alarmas, y la tabla de la sección 6 muestra ese intercambio.
- **Localización:** Grad-CAM sobre el logit de "grieta" en 8×8 y 16×16, TTA, refinamiento con mapa de crestas y cajas **solo si el clasificador detecta grieta**.
- **Prueba fuera del dataset:** sección 8b con imágenes externas.
- **Limitaciones:** el mapa de calor es más grueso que la grieta real y el refinamiento por crestas asume grietas oscuras; mejorarlo requeriría
  entrenar un modelo de segmentación con las máscaras pseudo-etiquetadas por Grad-CAM (siguiente paso natural). Las métricas de localización
  solo son reportables si se anotan máscaras a mano.